In [6]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------------------------------------- 8.9/8.9 MB 91.8 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- -------------

In [4]:
"""
Mock dataset generator for 14D malware feature vectors
"""
import numpy as np

np.random.seed(42)

NUM_SAMPLES = 2000

X = []
y = []

def benign_sample():
    return [
        np.random.randint(300, 2000), # length
        np.random.randint(10, 80), # lines
        np.random.uniform(10, 60), # avg line length
        np.random.uniform(0.05, 0.15), # digitRatio
        np.random.uniform(0.6, 0.8), # letterRatio
        np.random.uniform(0.1, 0.25), # symbolRatio
        np.random.uniform(3.5, 4.5), # entropy
        *np.random.binomial(1, 0.05, 7) # keyword counts
    ]
    
def malicious_sample():
    return [
        np.random.randint(1000, 10000),
        np.random.randint(1, 5),
        np.random.uniform(200, 2000),
        np.random.uniform(0.25, 0.45),
        np.random.uniform(0.1, 0.3),
        np.random.uniform(0.3, 0.6),
        np.random.uniform(5.0, 6.5),
        *np.random.binomial(3, 0.6, 7)
    ]
    
for _ in range(NUM_SAMPLES // 2):
    X.append(benign_sample())
    y.append(0)

for _ in range(NUM_SAMPLES // 2):
    X.append(malicious_sample())
    y.append(1)
    
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

In [6]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(14,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

model.save("saved_model")
model.save("model.h5")

Epoch 1/20
50/50 [==============================] - 1s 3ms/step - loss: 151.9398 - accuracy: 0.5013 - val_loss: 6.6500 - val_accuracy: 0.4625
Epoch 2/20
50/50 [==============================] - 0s 1ms/step - loss: 3.4605 - accuracy: 0.6662 - val_loss: 0.2015 - val_accuracy: 0.9300
Epoch 3/20
50/50 [==============================] - 0s 2ms/step - loss: 0.1245 - accuracy: 0.9525 - val_loss: 0.0561 - val_accuracy: 0.9925
Epoch 4/20
50/50 [==============================] - 0s 1ms/step - loss: 0.0627 - accuracy: 0.9837 - val_loss: 0.0623 - val_accuracy: 0.9900
Epoch 5/20
50/50 [==============================] - 0s 1ms/step - loss: 0.0524 - accuracy: 0.9875 - val_loss: 0.0324 - val_accuracy: 0.9975
Epoch 6/20
50/50 [==============================] - 0s 1ms/step - loss: 0.0517 - accuracy: 0.9894 - val_loss: 0.0314 - val_accuracy: 0.9975
Epoch 7/20
50/50 [==============================] - 0s 2ms/step - loss: 0.0477 - accuracy: 0.9894 - val_loss: 0.0228 - val_accuracy: 1.0000
Epoch 8/20
50/50 [

INFO:tensorflow:Assets written to: saved_model\assets
C:\Users\user\miniconda3\envs\tfjs_clean\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
